In [5]:
import TomatoSoup2 as tom
from experiment_management import ExperimentTemplateService, ExperimentLiveInstanceService
import pandas as pd
import textwrap
import json
def wrap(s, width=30):
    return "<br>".join(textwrap.wrap(str(s), width=width))

In [10]:
base_template = {
    "loss_name": tom.LOSS_NAME,
    "loss_additional_parameters": tom.LOSS_KWARGS,
    "optimizer_name": tom.OPTIMIZER_NAME,
    "optimizer_additional_parameters": tom.OPTIMIZER_KWARGS,
    "model_name": tom.MODEL_ID,
    "module_name": tom.MODULE_NAME,
}
temp_et = ExperimentTemplateService.create_non_persisted(normalization=1.0,**base_template)
temp_list = ExperimentTemplateService.find_matching(temp_et,excluded = ["group_id","batch_size","experiment_template_id"])
for temp in temp_list:
    print("new template")
    print(f"prompt group: {temp.prompt_group.group_id}")
    for prompt in temp.prompt_group.prompts:
        print(f"prompt: {prompt.prompt_id}")
        print(json.loads(prompt.text)[0]["content"][0]["text"])

new template
prompt group: 1
prompt: 1
Write a recipe for onion soup
new template
prompt group: 2
prompt: 2
Write a recipe for tomato soup


In [15]:
onion_soup_id = 1
tomato_soup_id = 2

In [ ]:
#getting the data into a single dataframe

base_template = {
    "loss_name": tom.LOSS_NAME,
    "loss_additional_parameters": tom.LOSS_KWARGS,
    "optimizer_name": tom.OPTIMIZER_NAME,
    "optimizer_additional_parameters": tom.OPTIMIZER_KWARGS,
    "model_name": tom.MODEL_ID,
    "module_name": tom.MODULE_NAME,
    "batch_size": 1
}


In [ ]:
def make_rows(group_id,prompt_name):
    temp_et = ExperimentTemplateService.create_non_persisted(group_id = group_id,**base_template)
    temp_list = ExperimentTemplateService.find_matching(temp_et,excluded = ["normalization","experiment_template_id"])
    temp_list = ExperimentTemplateService.find_matching(temp_et,excluded = ["normalization","experiment_template_id"])
    rows  = []

    for et in temp_list:
        live_instances = ExperimentLiveInstanceService.find_by({"experiment_template_id":et.experiment_template_id})
        norm = et.normalization
        snapshots = [sorted(live.snapshots,key=lambda snap: snap.iteration_count)[0] for live in live_instances]
        for live,snap in zip(live_instances,snapshots):
            vec_id = live.initial_vector_id
            steered_on_vanilla = list(filter(lambda metric: metric.description == "perplexity_steered_on_vanilla_generated_text",snap.metrics))[0].value
            steered_on_steered = list(filter(lambda metric: metric.description == "perplexity_steered_on_steered_generated_text",snap.metrics))[0].value
            vanilla_on_steered = list(filter(lambda metric: metric.description == "perplexity_vanilla_on_steered_generated_text",snap.generated_outputs[0].metrics))[0].value
            generated_text = wrap(snap.generated_outputs[0].text)
            row = {
                "vector":vec_id,
                "magnitude":norm,
                "steered on vanilla":steered_on_vanilla,
                "steered on steered": steered_on_steered,
                "vanilla on steered": vanilla_on_steered,
                "generated text": generated_text,
                "prompt":prompt_name
            }

            if norm < 9:rows.append(row) #remove the high magnitude ones as they are total nonsense
    return rows

In [37]:
tomato_rows = make_rows(tomato_soup_id,"tomato")
onion_rows = make_rows(onion_soup_id,"onion")
df = pd.DataFrame(tomato_rows + onion_rows)

In [45]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

y_cols = ['steered on vanilla','steered on steered','vanilla on steered']

titles = list([["tomato " + col] + ["onion " + col] for col in y_cols])
titles = [item for sublist in titles for item in sublist]

vectors = list(df['vector'].unique())
# ensure exactly 5 colors for 5 vectors
colors = {
    vectors[0]: "#1f77b4",  # blue
    vectors[1]: "#ff7f0e",  # orange
    vectors[2]: "#2ca02c",  # green
    vectors[3]: "#d62728",  # red
    vectors[4]: "#9467bd",  # purple
}

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# example: 2 columns, rows = len(y_cols)
rows = len(y_cols)
fig = make_subplots(rows=rows, cols=2,
                    subplot_titles=titles,shared_xaxes=True, shared_yaxes=False)

for i, name in enumerate(["tomato","onion"]):
    for row, y_col in enumerate(y_cols, start=1):
        for vec in vectors:
            d = df[(df['vector'] == vec) & (df["prompt"] == name)]
            fig.add_trace(
                go.Scatter(
                    x=d['magnitude'],
                    y=d[y_col],
                    mode='lines+markers',
                    name=f"vector {vec}",
                    legendgroup=str(vec),
                    line=dict(color=colors[vec]),
                    marker=dict(color=colors[vec]),
                    hovertext=d["generated text"],
                ),
                row=row, col=i+1
            )



fig.update_layout(height=900, xaxis_title='magnitude')
fig.show(renderer='browser')



Opening in existing browser session.
